# Notebook 08 — Stereo Vision

**Vision & 3D Mapping Workshop** | Block 3: Depth & 3D Reconstruction

---

## Why This Matters

Stereo vision is the oldest and most intuitive method for recovering 3-D structure from images.
Just as humans perceive depth by combining slightly different views from two eyes, a stereo
camera rig uses the horizontal disparity between left and right images to triangulate depth.

The pipeline is conceptually simple:

1. **Rectify** the image pair so that epipolar lines become horizontal scanlines.
2. **Match** corresponding pixels along scanlines to produce a **disparity map**.
3. **Convert** disparity to depth using the stereo equation $Z = fB/d$.
4. **Back-project** to 3-D using the camera intrinsics.

### What You'll Learn

1. **Stereo geometry** — two cameras, baseline, rectification
2. **The stereo equation** — full derivation of $Z = fB/d$ and depth precision $\Delta Z$
3. **Semi-Global Block Matching (SGBM)** — energy function, Census transform, 8-direction aggregation
4. **Disparity → depth → point cloud** — end-to-end reconstruction
5. **Depth precision analysis** — how baseline, focal length, and sub-pixel accuracy affect usable range
6. **Exercises** — synthetic stereo pair, disparity, depth, point cloud

### Prerequisites

| Concept | Where |
|:---|:---|
| Pinhole camera model, intrinsics $K$ | Notebook 03 |
| Rotation matrices, $\text{SO}(3)$ | Notebook 04 |
| Homogeneous coordinates, projection | Notebook 02 |
| Fundamental / Essential matrices | Notebook 05 |

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage import gaussian_filter

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "image.cmap": "gray",
    "axes.grid": False,
})

---
## 1. Stereo Geometry: Two Cameras, Baseline, Rectification

### 1.0 Historical Context

Stereo vision dates to Wheatstone's stereoscope (1838) — the first demonstration that
horizontal disparity between two images creates depth perception. The computational
formulation began with Marr & Poggio (1976), who proposed cooperative stereo matching
via inter-ocular disparity. The modern energy-minimisation framework (SGM/SGBM) was
introduced by Hirschmüller (2005, 2008), building on Scharstein & Szeliski's taxonomy
(2002) which established standard benchmarks that drive the field to this day.

### 1.1 The Stereo Rig

A stereo camera consists of two pinhole cameras separated by a known **baseline** $B$.
In the canonical (rectified) configuration the cameras share the same orientation and
differ only by a pure horizontal translation:

$$
P_L = K \begin{bmatrix} I & \mathbf{0} \end{bmatrix}, \qquad
P_R = K \begin{bmatrix} I & -B \mathbf{e}_1 \end{bmatrix}
$$

where $\mathbf{e}_1 = [1, 0, 0]^T$ and $K$ is the shared intrinsic matrix:

$$
K = \begin{bmatrix} f & 0 & c_x \\ 0 & f & c_y \\ 0 & 0 & 1 \end{bmatrix}
$$

### 1.2 Epipolar Geometry in the Stereo Case

For a general two-view setup, the epipolar constraint reads:

$$
\mathbf{x}_R^T \, F \, \mathbf{x}_L = 0
$$

The fundamental matrix $F$ maps a point in the left image to its epipolar line
in the right image. For a rectified stereo pair, $F$ simplifies dramatically:

$$
F_{\text{rect}} = [\mathbf{e}']_\times = \begin{bmatrix} 0 & 0 & 0 \\ 0 & 0 & -1 \\ 0 & 1 & 0 \end{bmatrix}
$$

This means every epipolar line is **horizontal**: the search for correspondences
reduces from 2-D to 1-D (same scanline row).

### 1.3 Rectification

Given an arbitrary stereo pair with extrinsics $(R, \mathbf{t})$, **stereo rectification**
finds rotation matrices $R_1, R_2$ such that after applying the homographies:

$$
H_L = K \, R_1 \, K^{-1}, \qquad H_R = K \, R_2 \, K^{-1}
$$

- Both image planes become **coplanar**
- Epipoles are sent to infinity → epipolar lines are horizontal
- The new $x$-axis aligns with the baseline: $\mathbf{e}_1 = \mathbf{t} / \|\mathbf{t}\|$

The rotation that achieves this sets the new camera axes as:

$$
\mathbf{r}_1 = \frac{\mathbf{t}}{\|\mathbf{t}\|}, \qquad
\mathbf{r}_2 = \frac{\mathbf{r}_3 \times \mathbf{r}_1}{\|\mathbf{r}_3 \times \mathbf{r}_1\|}, \qquad
\mathbf{r}_3 = \text{original optical axis}
$$

**Why this produces horizontal epipolar lines.** The epipole in each image is the
projection of the other camera's centre — equivalently, the vanishing point of the
baseline direction $\mathbf{t}$. Every epipolar line passes through the epipole.
By choosing $\mathbf{r}_1 = \mathbf{t}/\|\mathbf{t}\|$ as the new $x$-axis, the
baseline becomes a pure $x$-translation in the rectified frame, so the epipole is
projected to $(f \cdot \infty,\, 0)$ — it lies at infinity along the $x$-axis.
All lines through a point at infinity on the $x$-axis are horizontal, so every
epipolar line becomes a horizontal scanline. $\mathbf{r}_3$ is kept as the original
optical axis to minimise geometric distortion, and
$\mathbf{r}_2 = (\mathbf{r}_3 \times \mathbf{r}_1) / \|\mathbf{r}_3 \times \mathbf{r}_1\|$
simply completes a right-handed orthonormal basis for the new camera frame.

In [ ]:
def make_stereo_cameras(f, cx, cy, baseline):
    """Create canonical stereo camera matrices."""
    K = np.array([[f, 0, cx],
                  [0, f, cy],
                  [0, 0, 1]], dtype=np.float64)
    P_L = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P_R = K @ np.hstack([np.eye(3), np.array([[-baseline], [0], [0]])])
    return K, P_L, P_R


f = 500.0
cx, cy = 320.0, 240.0
B = 0.12  # 12 cm baseline
K, P_L, P_R = make_stereo_cameras(f, cx, cy, B)

print("Intrinsic matrix K:")
print(K)
print(f"\nBaseline B = {B} m")
print(f"\nP_L =\n{P_L}")
print(f"\nP_R =\n{P_R}")

In [ ]:
def visualize_stereo_rig(K, B):
    """Visualize the stereo camera rig in 3D."""
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    f = K[0, 0]
    scale = 0.05
    
    for cam_x, label, color in [(0, 'Left', 'blue'), (B, 'Right', 'red')]:
        cam_pos = np.array([cam_x, 0, 0])
        corners = np.array([
            [cam_x - scale, -scale, scale * f / 100],
            [cam_x + scale, -scale, scale * f / 100],
            [cam_x + scale,  scale, scale * f / 100],
            [cam_x - scale,  scale, scale * f / 100],
        ])
        for c in corners:
            ax.plot([cam_pos[0], c[0]], [cam_pos[1], c[1]], [cam_pos[2], c[2]],
                    color=color, linewidth=1)
        verts = [corners.tolist()]
        poly = Poly3DCollection(verts, alpha=0.15, facecolor=color)
        ax.add_collection3d(poly)
        ax.scatter(*cam_pos, s=60, c=color, marker='o', zorder=5)
        ax.text(cam_pos[0], cam_pos[1] - 0.03, cam_pos[2], label,
                fontsize=10, color=color)
    
    ax.plot([0, B], [-0.06, -0.06], [0, 0], color='green', lw=2)
    ax.text(B/2, -0.08, 0, f'B = {B} m', fontsize=11, color='green', ha='center')
    
    scene_pts = np.array([[0.0, 0.0, 3.0], [-0.5, 0.3, 4.0], [0.3, -0.2, 2.5]])
    ax.scatter(scene_pts[:, 0], scene_pts[:, 1], scene_pts[:, 2],
               s=80, c='black', marker='^', label='Scene points')
    for pt in scene_pts:
        ax.plot([0, pt[0]], [0, pt[1]], [0, pt[2]], 'b--', alpha=0.3)
        ax.plot([B, pt[0]], [0, pt[1]], [0, pt[2]], 'r--', alpha=0.3)
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title('Stereo Camera Rig with Scene Points')
    ax.legend()
    plt.tight_layout()
    plt.show()

visualize_stereo_rig(K, B)

---
## 2. The Stereo Equation: Derivation from Similar Triangles

### 2.1 Setup

Consider a 3-D point $\mathbf{P} = (X, Y, Z)^T$ observed by two cameras with:

- Focal length $f$ (in pixels)
- Baseline $B$ (in metres)
- Left camera at the origin, right camera at $(B, 0, 0)$

The left camera projects $\mathbf{P}$ to:

$$
x_L = f \frac{X}{Z} + c_x, \qquad y_L = f \frac{Y}{Z} + c_y
$$

The right camera, shifted by $B$ along the $x$-axis, projects to:

$$
x_R = f \frac{X - B}{Z} + c_x, \qquad y_R = f \frac{Y}{Z} + c_y
$$

### 2.2 Disparity

The **disparity** $d$ is the horizontal pixel difference:

$$
d = x_L - x_R = f \frac{X}{Z} - f \frac{X - B}{Z} = f \frac{X - (X - B)}{Z} = \frac{fB}{Z}
$$

### 2.3 The Stereo Depth Equation

Solving for depth:

$$
\boxed{Z = \frac{f \cdot B}{d}}
$$

This is the **fundamental equation of stereo vision**. Key properties:

- Depth is **inversely proportional** to disparity
- Larger baselines → larger disparities → better depth resolution at distance
- Zero disparity → infinite depth

### 2.4 Depth Precision

Differentiating the stereo equation $Z = fB/d$ (§2.3) with respect to disparity $d$:

$$
\frac{dZ}{dd} = \frac{d}{dd}\left(\frac{fB}{d}\right) = fB \cdot (-d^{-2}) = -\frac{fB}{d^2}
$$

Substituting $d = fB/Z$:

$$
\frac{dZ}{dd} = -\frac{fB}{(fB/Z)^2} = -\frac{fB \cdot Z^2}{f^2 B^2} = -\frac{Z^2}{fB}
$$

The absolute depth error for a disparity uncertainty $\Delta d$ is therefore:

$$
\boxed{\Delta Z = \frac{Z^2}{f \cdot B} \cdot \Delta d}
$$

**Critical insight**: depth error grows **quadratically** with distance $Z$.

| Distance $Z$ | $\Delta d = 0.5$ px, $f=500$, $B=0.12$ m | $\Delta Z$ |
|:---:|:---:|:---:|
| 1 m | $1^2/(500 \times 0.12) \times 0.5$ | 0.008 m |
| 5 m | $5^2/(500 \times 0.12) \times 0.5$ | 0.21 m |
| 10 m | $10^2/(500 \times 0.12) \times 0.5$ | 0.83 m |
| 50 m | $50^2/(500 \times 0.12) \times 0.5$ | 20.8 m |

### 2.5 3-D Back-projection and the Q Matrix

Once we know $Z$, we recover the full 3-D point:

$$
X = \frac{(u - c_x) \cdot Z}{f}, \qquad Y = \frac{(v - c_y) \cdot Z}{f}
$$

#### Row-by-row derivation of the reprojection matrix Q

OpenCV's `cv2.reprojectImageTo3D` uses the matrix $Q$ to map
$(u, v, d, 1)$ directly to homogeneous 3-D coordinates. We derive $Q$
row by row from the stereo equations.

**Starting equations** (§2.1–2.3):

$$
X_w = \frac{(u - c_x) \cdot Z}{f}, \qquad
Y_w = \frac{(v - c_y) \cdot Z}{f}, \qquad
Z_w = \frac{fB}{d}
$$

We seek $Q$ such that:

$$
\begin{bmatrix} X \\ Y \\ Z \\ W \end{bmatrix}
= Q \begin{bmatrix} u \\ v \\ d \\ 1 \end{bmatrix}, \qquad
\text{with } (X_w, Y_w, Z_w) = (X/W,\; Y/W,\; Z/W)
$$

Choose $W = d/B$ — the simplest expression linear in $d$ that makes every row
work. Substituting $Z_w = fB/d$ into each coordinate:

**Row 4** ($W$): We need $W = d/B = 0 \cdot u + 0 \cdot v + \tfrac{1}{B} \cdot d + 0$

$$\Rightarrow \text{Row 4} = \begin{bmatrix} 0 & 0 & 1/B & 0 \end{bmatrix}$$

**Row 3** ($Z$): $Z = W \cdot Z_w = \dfrac{d}{B} \cdot \dfrac{fB}{d} = f$, a constant independent of $(u,v,d)$

$$\Rightarrow \text{Row 3} = \begin{bmatrix} 0 & 0 & 0 & f \end{bmatrix}$$

**Row 1** ($X$): $X = W \cdot X_w = \dfrac{d}{B} \cdot \dfrac{(u - c_x)\,B}{d} = u - c_x$

$$\Rightarrow \text{Row 1} = \begin{bmatrix} 1 & 0 & 0 & -c_x \end{bmatrix}$$

**Row 2** ($Y$): $Y = W \cdot Y_w = \dfrac{d}{B} \cdot \dfrac{(v - c_y)\,B}{d} = v - c_y$

$$\Rightarrow \text{Row 2} = \begin{bmatrix} 0 & 1 & 0 & -c_y \end{bmatrix}$$

Assembling:

$$
Q = \begin{bmatrix}
1 & 0 & 0 & -c_x \\
0 & 1 & 0 & -c_y \\
0 & 0 & 0 & f \\
0 & 0 & 1/B & 0
\end{bmatrix}
$$

**Verification:** $Q [u,\, v,\, d,\, 1]^T = [u - c_x,\; v - c_y,\; f,\; d/B]^T$, giving:

$$
X_w = \frac{u - c_x}{d/B} = \frac{(u - c_x)\,B}{d}, \quad
Y_w = \frac{v - c_y}{d/B} = \frac{(v - c_y)\,B}{d}, \quad
Z_w = \frac{f}{d/B} = \frac{fB}{d} \;\checkmark
$$

OpenCV's actual $Q$ matrix uses $T_x$ (the signed $x$-translation, typically
negative) in place of $B$, so the sign of the $(3,2)$ entry depends on the
rig convention.

In [ ]:
def stereo_depth(f, B, d):
    """Z = f*B/d"""
    return f * B / d

def depth_precision(Z, f, B, delta_d):
    """ΔZ = Z²/(f·B) · Δd"""
    return (Z**2 / (f * B)) * delta_d

print("=== Stereo Equation Verification ===")
test_Z = 5.0
d_expected = f * B / test_Z
Z_recovered = stereo_depth(f, B, d_expected)
print(f"True depth Z = {test_Z} m")
print(f"Expected disparity d = fB/Z = {f}×{B}/{test_Z} = {d_expected:.2f} px")
print(f"Recovered depth Z = fB/d = {f}×{B}/{d_expected:.2f} = {Z_recovered:.2f} m")

print("\n=== Depth Precision ===")
delta_d = 0.5
for Z_test in [1, 5, 10, 20, 50]:
    dZ = depth_precision(Z_test, f, B, delta_d)
    print(f"Z = {Z_test:>2d} m  →  ΔZ = {dZ:.4f} m  ({dZ/Z_test*100:.2f}% relative error)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

d_range = np.linspace(0.5, 200, 500)
for B_val in [0.06, 0.12, 0.24, 0.54]:
    Z_vals = stereo_depth(f, B_val, d_range)
    ax1.plot(d_range, Z_vals, label=f'B = {B_val} m')
ax1.set_xlabel('Disparity d (pixels)')
ax1.set_ylabel('Depth Z (metres)')
ax1.set_title(r'Stereo Equation: $Z = fB/d$')
ax1.set_ylim(0, 50)
ax1.legend()
ax1.grid(True, alpha=0.3)

Z_range = np.linspace(0.5, 60, 500)
delta_d = 0.5
for B_val in [0.06, 0.12, 0.24, 0.54]:
    dZ_vals = depth_precision(Z_range, f, B_val, delta_d)
    ax2.semilogy(Z_range, dZ_vals, label=f'B = {B_val} m')
ax2.set_xlabel('Distance Z (metres)')
ax2.set_ylabel(r'$\Delta Z$ (metres)')
ax2.set_title(r'Depth Precision: $\Delta Z = Z^2 \cdot \Delta d \,/\, (fB)$')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1.0, color='k', linestyle='--', alpha=0.3, label='1 m error')

plt.tight_layout()
plt.show()

---
## 3. SGBM Disparity Computation

### 3.1 The Stereo Matching Problem

Given rectified images $I_L$ and $I_R$, find for each pixel $(x, y)$ in the left
image the disparity $d(x, y) \geq 0$ such that:

$$
I_L(x, y) \approx I_R(x - d, y)
$$

### 3.2 Energy Function

SGBM frames matching as an energy minimisation problem over the disparity image $D$:

$$
E(D) = \sum_{\mathbf{p}} \Bigl[
  C\bigl(\mathbf{p}, D(\mathbf{p})\bigr)
  + \sum_{\mathbf{q} \in \mathcal{N}(\mathbf{p})} P_1 \cdot \mathbb{1}\bigl[|D(\mathbf{p}) - D(\mathbf{q})| = 1\bigr]
  + \sum_{\mathbf{q} \in \mathcal{N}(\mathbf{p})} P_2 \cdot \mathbb{1}\bigl[|D(\mathbf{p}) - D(\mathbf{q})| > 1\bigr]
\Bigr]
$$

where:

| Term | Meaning |
|:---|:---|
| $C(\mathbf{p}, d)$ | Pixel-wise matching cost at pixel $\mathbf{p}$ for disparity $d$ |
| $P_1$ | Penalty for 1-pixel disparity change (smooth surfaces) |
| $P_2$ | Penalty for larger jumps ($P_2 > P_1$, preserves depth edges) |
| $\mathcal{N}(\mathbf{p})$ | 4-connected neighbourhood of pixel $\mathbf{p}$ |

The two-penalty design lets SGBM reconstruct both smooth surfaces (where $P_1$ dominates) and sharp depth boundaries at object edges (where $P_2$ permits large jumps) — critical for obstacle detection where depth discontinuities mark object boundaries.

### 3.3 Census Transform

The **Census transform** encodes local structure as a bit string:

$$
\text{Census}(\mathbf{p}) = \bigotimes_{\mathbf{q} \in W(\mathbf{p})}
\mathbb{1}\bigl[I(\mathbf{q}) < I(\mathbf{p})\bigr]
$$

For a $7 \times 7$ window, this produces a 48-bit descriptor. The matching cost
between two Census descriptors is the **Hamming distance** (number of differing bits):

$$
C_{\text{census}}(\mathbf{p}, d) = \text{popcount}\bigl(
  \text{Census}_L(\mathbf{p}) \oplus \text{Census}_R(\mathbf{p} - d)
\bigr)
$$

### 3.4 Semi-Global Aggregation (8 Directions)

Minimising $E(D)$ globally is NP-hard (it's a 2-D MRF). SGBM approximates the
global solution by aggregating **1-D dynamic programming** solutions along
$r = 8$ directions (horizontal, vertical, and both diagonals).

For each direction $\mathbf{r}$, define the path cost:

$$
L_{\mathbf{r}}(\mathbf{p}, d) = C(\mathbf{p}, d) + \min\begin{cases}
L_{\mathbf{r}}(\mathbf{p} - \mathbf{r}, d) \\
L_{\mathbf{r}}(\mathbf{p} - \mathbf{r}, d-1) + P_1 \\
L_{\mathbf{r}}(\mathbf{p} - \mathbf{r}, d+1) + P_1 \\
\min_i L_{\mathbf{r}}(\mathbf{p} - \mathbf{r}, i) + P_2
\end{cases}
- \min_k L_{\mathbf{r}}(\mathbf{p} - \mathbf{r}, k)
$$

The subtraction of $\min_k L_{\mathbf{r}}(\mathbf{p}-\mathbf{r}, k)$ prevents the
cost from growing unboundedly along the path. **Why the argmin is preserved:**
$\min_k L_{\mathbf{r}}(\mathbf{p}-\mathbf{r}, k)$ depends only on the *previous*
pixel along the path — it is a constant with respect to the current disparity $d$.
Subtracting the same constant from every candidate disparity at pixel $\mathbf{p}$
shifts all path costs by the same amount, so
$\arg\min_d L_{\mathbf{r}}(\mathbf{p}, d)$ is unchanged. Without this normalisation,
$L_{\mathbf{r}}$ accumulates matching costs along the path and can overflow for long
scanlines.

The final aggregated cost:

$$
S(\mathbf{p}, d) = \sum_{\mathbf{r}} L_{\mathbf{r}}(\mathbf{p}, d)
$$

The winning disparity at each pixel is:

$$
D(\mathbf{p}) = \arg\min_d S(\mathbf{p}, d)
$$

Sub-pixel refinement uses a quadratic fit to the cost curve around the minimum.

In [ ]:
def census_transform(img, window_size=7):
    """Census transform: encode local intensity ordering as bit string."""
    h, w = img.shape
    half = window_size // 2
    census = np.zeros((h, w), dtype=np.uint64)
    
    for dy in range(-half, half + 1):
        for dx in range(-half, half + 1):
            if dy == 0 and dx == 0:
                continue
            shifted = np.zeros_like(img)
            y1s, y1e = max(0, dy), min(h, h + dy)
            x1s, x1e = max(0, dx), min(w, w + dx)
            y2s, y2e = max(0, -dy), min(h, h - dy)
            x2s, x2e = max(0, -dx), min(w, w - dx)
            shifted[y2s:y2e, x2s:x2e] = img[y1s:y1e, x1s:x1e]
            census = (census << 1) | (shifted < img).astype(np.uint64)
    
    return census


def hamming_distance_map(census_l, census_r, d):
    """Compute pixel-wise Hamming distance for disparity d."""
    h, w = census_l.shape
    cost = np.full((h, w), 48, dtype=np.float32)
    if d < w:
        xor = census_l[:, d:] ^ census_r[:, :w - d]
        bits = np.zeros((h, w - d), dtype=np.int32)
        temp = xor.copy()
        while np.any(temp > 0):
            bits += (temp & 1).astype(np.int32)
            temp >>= 1
        cost[:, d:] = bits.astype(np.float32)
    return cost


print("For a 7×7 window: 48-bit descriptor per pixel.")
print(f"P1 = 8 × blockSize² = 8 × 5² = {8 * 25}")
print(f"P2 = 32 × blockSize² = 32 × 5² = {32 * 25}")

In [ ]:
directions = [
    ( 0,  1),  # right
    ( 0, -1),  # left
    ( 1,  0),  # down
    (-1,  0),  # up
    ( 1,  1),  # down-right
    ( 1, -1),  # down-left
    (-1,  1),  # up-right
    (-1, -1),  # up-left
]

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
colors = plt.cm.Set1(np.linspace(0, 1, 8))
for i, (dy, dx) in enumerate(directions):
    ax.annotate('', xy=(dx * 1.5, -dy * 1.5), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
    ax.text(dx * 1.7, -dy * 1.7, f'r{i+1}', fontsize=10, ha='center', va='center',
            color=colors[i], fontweight='bold')
ax.plot(0, 0, 'ko', markersize=8)
ax.set_title('SGBM 8-Direction Aggregation Paths', fontsize=13)
ax.grid(True, alpha=0.2)
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.tight_layout()
plt.show()

---
## 4. Synthetic Stereo Pair and Disparity Computation

We generate a synthetic 3-D scene and render left/right views to test the full
stereo pipeline. The scene contains planar surfaces at different depths and
textured regions for reliable matching.

In [ ]:
def generate_synthetic_scene(H=480, W=640):
    """Generate a synthetic depth map with textured objects at various depths."""
    depth_gt = np.full((H, W), 10.0, dtype=np.float32)
    
    depth_gt[200:400, 100:300] = 3.0
    depth_gt[150:350, 350:550] = 5.0
    depth_gt[50:150, 200:450] = 7.0
    depth_gt[350:450, 400:600] = 2.0
    
    return depth_gt


def render_textured_image(H=480, W=640, seed=42):
    """Render a textured image with random patterns."""
    rng = np.random.RandomState(seed)
    noise = rng.randint(0, 255, (H // 4, W // 4), dtype=np.uint8)
    img = cv2.resize(noise, (W, H), interpolation=cv2.INTER_LINEAR)
    img = gaussian_filter(img.astype(np.float32), sigma=1.5)
    
    for cx_rect, cy_rect, rw, rh, val in [
        (200, 300, 100, 100, 180), (450, 250, 100, 100, 80),
        (325, 100, 125, 50, 200), (500, 400, 100, 50, 60),
    ]:
        x1, x2 = cx_rect - rw, cx_rect + rw
        y1, y2 = cy_rect - rh, cy_rect + rh
        x1, x2 = max(0, x1), min(W, x2)
        y1, y2 = max(0, y1), min(H, y2)
        local_noise = rng.randint(0, 40, (y2 - y1, x2 - x1)).astype(np.float32)
        img[y1:y2, x1:x2] = val + local_noise
    
    return np.clip(img, 0, 255).astype(np.uint8)


def create_stereo_pair(img_left, depth_gt, f, B):
    """Warp left image to create right image using ground-truth depth."""
    H, W = img_left.shape[:2]
    disparity_gt = np.zeros_like(depth_gt)
    valid = depth_gt > 0
    disparity_gt[valid] = f * B / depth_gt[valid]
    
    img_right = np.zeros_like(img_left)
    for y in range(H):
        for x in range(W):
            d = disparity_gt[y, x]
            x_r = int(round(x - d))
            if 0 <= x_r < W:
                img_right[y, x_r] = img_left[y, x]
    
    mask = img_right == 0
    if mask.any():
        from scipy.ndimage import grey_dilation
        img_right_filled = grey_dilation(img_right, size=3)
        img_right[mask] = img_right_filled[mask]
    
    return img_right, disparity_gt


H, W = 480, 640
depth_gt = generate_synthetic_scene(H, W)
img_left = render_textured_image(H, W, seed=42)
img_right, disparity_gt = create_stereo_pair(img_left, depth_gt, f, B)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].imshow(img_left, cmap='gray')
axes[0, 0].set_title('Left Image')
axes[0, 1].imshow(img_right, cmap='gray')
axes[0, 1].set_title('Right Image')
im2 = axes[1, 0].imshow(disparity_gt, cmap='turbo')
axes[1, 0].set_title('Ground Truth Disparity')
plt.colorbar(im2, ax=axes[1, 0], label='pixels')
im3 = axes[1, 1].imshow(depth_gt, cmap='turbo_r')
axes[1, 1].set_title('Ground Truth Depth')
plt.colorbar(im3, ax=axes[1, 1], label='metres')
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Synthetic Stereo Pair', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from src.stereo import compute_disparity_sgbm, disparity_to_depth

disparity_est = compute_disparity_sgbm(img_left, img_right,
                                        num_disparities=128, block_size=5)
depth_est = disparity_to_depth(disparity_est, f, B)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
im0 = axes[0, 0].imshow(disparity_gt, cmap='turbo')
axes[0, 0].set_title('GT Disparity')
plt.colorbar(im0, ax=axes[0, 0], label='px')

im1 = axes[0, 1].imshow(disparity_est, cmap='turbo')
axes[0, 1].set_title('SGBM Disparity')
plt.colorbar(im1, ax=axes[0, 1], label='px')

im2 = axes[1, 0].imshow(depth_gt, cmap='turbo_r', vmin=1, vmax=12)
axes[1, 0].set_title('GT Depth')
plt.colorbar(im2, ax=axes[1, 0], label='m')

depth_est_clipped = np.clip(depth_est, 0, 15)
im3 = axes[1, 1].imshow(depth_est_clipped, cmap='turbo_r', vmin=1, vmax=12)
axes[1, 1].set_title('Estimated Depth')
plt.colorbar(im3, ax=axes[1, 1], label='m')

for ax in axes.flat:
    ax.axis('off')
plt.suptitle('SGBM Stereo Matching Results', fontsize=14)
plt.tight_layout()
plt.show()

valid_mask = (disparity_est > 0) & (depth_gt > 0) & (depth_est > 0) & (depth_est < 15)
if valid_mask.any():
    depth_err = np.abs(depth_est[valid_mask] - depth_gt[valid_mask])
    print(f"Depth error stats (valid pixels: {valid_mask.sum()}):")
    print(f"  Mean:   {depth_err.mean():.3f} m")
    print(f"  Median: {np.median(depth_err):.3f} m")
    print(f"  Std:    {depth_err.std():.3f} m")
    print(f"  Max:    {depth_err.max():.3f} m")

### 4.5 Disparity Confidence & Left-Right Consistency Check

Not all disparity values are reliable. Three standard validation methods:

**1. Left-Right Consistency Check**

Compute disparity from *both* directions:
- $d_L(u, v)$: disparity of left image (matching left→right)
- $d_R(u, v)$: disparity of right image (matching right→left)

A pixel passes the consistency check if:

$$
|d_L(u, v) - d_R(u - d_L(u,v), v)| < \tau
$$

where $\tau$ is typically 1 pixel. **Intuition**: if pixel $u$ in the left image
matches to $u - d$ in the right image, then pixel $u - d$ in the right image should
match back to $u$ in the left image. Disagreement indicates occlusion or matching
failure.

**2. Texture Check**

Low-texture regions (walls, sky) produce unreliable disparity. Reject pixels where
the local variance of the reference image is below a threshold:
$\text{Var}(I_L(w)) < \sigma_{\min}^2$, where $w$ is a local window.

**3. Uniqueness Ratio**

If the best matching cost is not significantly better than the second-best,
the match is ambiguous. OpenCV's SGBM implements this via the `uniquenessRatio`
parameter: reject if $\text{cost}_{\text{best}} \cdot (1 + \text{ratio}/100) \geq \text{cost}_{\text{2nd}}$.

In OpenCV's SGBM, invalid disparities are marked as $-16$ (or the minimum disparity
minus one). Always filter these before converting to depth.

---
## 5. Disparity → Depth → Point Cloud

The final step converts the depth map into a 3-D point cloud. For each pixel
$(u, v)$ with valid depth $Z$:

$$
\mathbf{P}_{3D} = \begin{bmatrix}
\frac{(u - c_x) \cdot Z}{f_x} \\
\frac{(v - c_y) \cdot Z}{f_y} \\
Z
\end{bmatrix}
$$

In [ ]:
def depth_to_pointcloud(depth, K, max_depth=15.0):
    """Convert depth map to 3D point cloud."""
    H, W = depth.shape
    fx, fy = K[0, 0], K[1, 1]
    cx_k, cy_k = K[0, 2], K[1, 2]
    
    u, v = np.meshgrid(np.arange(W), np.arange(H))
    valid = (depth > 0) & (depth < max_depth)
    
    z = depth[valid]
    x = (u[valid] - cx_k) * z / fx
    y = (v[valid] - cy_k) * z / fy
    
    return np.stack([x, y, z], axis=-1)


points_gt = depth_to_pointcloud(depth_gt, K)
points_est = depth_to_pointcloud(depth_est, K)

fig = plt.figure(figsize=(18, 10))

# --- Row 1: 3D point clouds side by side ---
ax1 = fig.add_subplot(221, projection='3d')
subsample = np.random.choice(len(points_gt), min(5000, len(points_gt)), replace=False)
pts = points_gt[subsample]
ax1.scatter(pts[:, 0], pts[:, 2], -pts[:, 1], c=pts[:, 2], cmap='turbo_r', s=1, alpha=0.5)
ax1.set_xlabel('X [m]'); ax1.set_ylabel('Z [m]'); ax1.set_zlabel('-Y [m]')
ax1.set_title('Ground Truth Point Cloud')
ax1.view_init(elev=20, azim=-55)

ax2 = fig.add_subplot(222, projection='3d')
if len(points_est) > 0:
    subsample_e = np.random.choice(len(points_est), min(5000, len(points_est)), replace=False)
    pts_e = points_est[subsample_e]
    ax2.scatter(pts_e[:, 0], pts_e[:, 2], -pts_e[:, 1], c=pts_e[:, 2], cmap='turbo_r', s=1, alpha=0.5)
ax2.set_xlabel('X [m]'); ax2.set_ylabel('Z [m]'); ax2.set_zlabel('-Y [m]')
ax2.set_title('Estimated Point Cloud (SGBM)')
ax2.view_init(elev=20, azim=-55)

# --- Row 2: Depth comparison + error map ---
ax3 = fig.add_subplot(223)
ax3.imshow(depth_gt, cmap='turbo', vmin=0, vmax=12)
ax3.set_title('Ground Truth Depth')
ax3.set_xlabel('u [px]'); ax3.set_ylabel('v [px]')

ax4 = fig.add_subplot(224)
depth_err = np.abs(depth_est - depth_gt)
depth_err[depth_est <= 0] = np.nan
im4 = ax4.imshow(depth_err, cmap='hot', vmin=0, vmax=2)
plt.colorbar(im4, ax=ax4, label='|Depth error| [m]')
ax4.set_title('Absolute Depth Error')
ax4.set_xlabel('u [px]'); ax4.set_ylabel('v [px]')

plt.suptitle('Stereo Pipeline: Disparity → Depth → 3D Point Cloud',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

valid_mask = (depth_est > 0) & (depth_gt > 0) & (depth_gt < 15)
if valid_mask.any():
    rmse = np.sqrt(np.mean((depth_est[valid_mask] - depth_gt[valid_mask])**2))
    mae = np.mean(np.abs(depth_est[valid_mask] - depth_gt[valid_mask]))
    print(f"GT points:  {len(points_gt):>6d},  Est points: {len(points_est):>6d}")
    print(f"Depth RMSE: {rmse:.3f} m,  MAE: {mae:.3f} m")
else:
    print(f"GT points:  {len(points_gt):>6d},  Est points: {len(points_est):>6d}")

---
## 4b. Real Stereo Data (Middlebury)

The [Middlebury Stereo Evaluation](https://vision.middlebury.edu/stereo/) provides high-quality rectified image pairs with known calibration. We load a local pair from `data/sample_pairs/middlebury/` when available, attempt to download the matching right view (`im1.png`), and fall back to a shifted synthetic right view if the download is blocked.

**Workflow:** load left → resolve right (local file / download / synthetic) → SGBM disparity → depth → coloured 3D point cloud.

In [ ]:
from pathlib import Path
import urllib.request

from src.stereo import StereoDepthEstimator

# Middlebury 2014 rectified calibration (Adirondack / Piano family, full resolution)
MIDDLEBURY_F = 3983.209
MIDDLEBURY_CX = 767.067
MIDDLEBURY_CY = 998.388
MIDDLEBURY_BASELINE_M = 0.176252
MIDDLEBURY_SCENES = ("Adirondack", "Piano", "Playroom")
MIDDLEBURY_DOWNLOAD_BASE = (
    "https://vision.middlebury.edu/stereo/data/scenes2014/datasets"
)


def _is_valid_png(path: Path) -> bool:
    if not path.is_file() or path.stat().st_size < 256:
        return False
    return path.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"


def _download_middlebury_image(scene: str, im_name: str, out_path: Path) -> bool:
    url = f"{MIDDLEBURY_DOWNLOAD_BASE}/{scene}-perfect/{im_name}"
    try:
        urllib.request.urlretrieve(url, out_path)
    except Exception as exc:
        print(f"  Download failed ({url}): {exc}")
        return False
    if not _is_valid_png(out_path):
        out_path.unlink(missing_ok=True)
        return False
    print(f"  Downloaded {out_path.name} ({out_path.stat().st_size // 1024} KB)")
    return True


def _make_synthetic_right(left_gray: np.ndarray, disparity_px: int = 40) -> np.ndarray:
    """Approximate right view by horizontal shift (fallback when im1.png unavailable)."""
    h, w = left_gray.shape
    d = int(np.clip(disparity_px, 8, w // 4))
    right = np.zeros_like(left_gray)
    right[:, : w - d] = left_gray[:, d:]
    if right.max() == 0:
        return left_gray.copy()
    mask = right == 0
    if mask.any():
        right_filled = cv2.GaussianBlur(right, (5, 5), 0)
        right[mask] = right_filled[mask]
    return right


def load_middlebury_stereo_pair(
    data_dir: Path | None = None,
    max_width: int = 960,
) -> dict | None:
    data_dir = data_dir or (Path("..") / "data" / "sample_pairs" / "middlebury")
    data_dir = data_dir.resolve()

    for scene in MIDDLEBURY_SCENES:
        left_path = data_dir / f"{scene}_left.png"
        right_path = data_dir / f"{scene}_right.png"
        if not _is_valid_png(left_path):
            continue

        right_source = "local"
        if not _is_valid_png(right_path):
            print(f"Right view missing for {scene}; trying download (im1.png)...")
            if _download_middlebury_image(scene, "im1.png", right_path):
                right_source = "downloaded"
            else:
                print("  Using synthetic right view (horizontal shift fallback).")
                right_source = "synthetic"

        left_bgr = cv2.imread(str(left_path))
        if right_source == "synthetic":
            left_gray_full = cv2.cvtColor(left_bgr, cv2.COLOR_BGR2GRAY)
            right_gray_full = _make_synthetic_right(left_gray_full, disparity_px=45)
            right_bgr = cv2.cvtColor(right_gray_full, cv2.COLOR_GRAY2BGR)
        else:
            right_bgr = cv2.imread(str(right_path))

        if left_bgr is None or right_bgr is None:
            continue

        h0, w0 = left_bgr.shape[:2]
        scale = 1.0
        if w0 > max_width:
            scale = max_width / w0
            new_size = (int(w0 * scale), int(h0 * scale))
            left_bgr = cv2.resize(left_bgr, new_size, interpolation=cv2.INTER_AREA)
            right_bgr = cv2.resize(right_bgr, new_size, interpolation=cv2.INTER_AREA)

        h, w = left_bgr.shape[:2]
        fx = fy = MIDDLEBURY_F * scale
        cx = MIDDLEBURY_CX * scale
        cy = MIDDLEBURY_CY * scale
        K_mb = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)

        return {
            "scene": scene,
            "left": left_bgr,
            "right": right_bgr,
            "K": K_mb,
            "baseline": MIDDLEBURY_BASELINE_M,
            "right_source": right_source,
            "scale": scale,
        }

    return None


mb = load_middlebury_stereo_pair()

if mb is None:
    print("No valid Middlebury left image found in data/sample_pairs/middlebury/.")
    print("Run data/download.sh or place Adirondack_left.png / Piano_left.png there.")
else:
    print(
        f"Middlebury scene: {mb['scene']}  |  right view: {mb['right_source']}  |  "
        f"scale={mb['scale']:.3f}"
    )

    img_left_mb = cv2.cvtColor(mb["left"], cv2.COLOR_BGR2GRAY)
    img_right_mb = cv2.cvtColor(mb["right"], cv2.COLOR_BGR2GRAY)
    K_mb, B_mb = mb["K"], mb["baseline"]

    disparity_mb = compute_disparity_sgbm(
        img_left_mb,
        img_right_mb,
        num_disparities=256,
        block_size=7,
    )
    depth_mb = disparity_to_depth(disparity_mb, K_mb[0, 0], B_mb)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes[0, 0].imshow(cv2.cvtColor(mb["left"], cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title(f"{mb['scene']} — Left")
    axes[0, 1].imshow(cv2.cvtColor(mb["right"], cv2.COLOR_BGR2RGB))
    axes[0, 1].set_title(f"Right ({mb['right_source']})")
    im_d = axes[0, 2].imshow(disparity_mb, cmap="turbo")
    axes[0, 2].set_title("SGBM Disparity")
    plt.colorbar(im_d, ax=axes[0, 2], fraction=0.046)

    depth_show = np.clip(depth_mb, 0, 8)
    im_z = axes[1, 0].imshow(depth_show, cmap="turbo_r", vmin=0.5, vmax=8)
    axes[1, 0].set_title("Estimated Depth (m)")
    plt.colorbar(im_z, ax=axes[1, 0], fraction=0.046)

    valid = disparity_mb > 0
    if valid.any():
        print(
            f"Disparity range: [{disparity_mb[valid].min():.1f}, {disparity_mb[valid].max():.1f}] px"
        )
        print(
            f"Depth range:     [{depth_mb[valid].min():.2f}, {depth_mb[valid].max():.2f}] m"
        )

    for ax in axes.flat:
        ax.axis("off")

    # 3D point cloud from real (or synthetic-right) stereo pair
    estimator = StereoDepthEstimator(K_mb, B_mb, (mb["left"].shape[1], mb["left"].shape[0]))
    points_mb, colors_mb = estimator.compute_pointcloud(mb["left"], depth_mb, max_depth=8.0)

    ax_pc = fig.add_subplot(2, 3, 5, projection="3d")
    stride = max(1, len(points_mb) // 8000) if len(points_mb) else 1
    if len(points_mb) > 0:
        pts = points_mb[::stride]
        cols = colors_mb[::stride] / 255.0
        ax_pc.scatter(
            pts[:, 0], pts[:, 2], -pts[:, 1], c=cols, s=0.5, alpha=0.6, linewidths=0
        )
    ax_pc.set_title("3D Point Cloud (Middlebury)")
    ax_pc.set_xlabel("X (m)")
    ax_pc.set_ylabel("Z (m)")
    ax_pc.set_zlabel("-Y (m)")

    ax_side = fig.add_subplot(2, 3, 6, projection="3d")
    if len(points_mb) > 0:
        pts = points_mb[::stride]
        ax_side.scatter(
            pts[:, 0], pts[:, 2], -pts[:, 1], c=pts[:, 2], cmap="turbo_r", s=0.5, alpha=0.6
        )
    ax_side.set_title("Point Cloud — depth coloured")
    ax_side.view_init(elev=5, azim=0)
    ax_side.set_xlabel("X")
    ax_side.set_ylabel("Z")
    ax_side.set_zlabel("-Y")

    plt.suptitle(
        f"Real Stereo Data: Middlebury {mb['scene']} (right: {mb['right_source']})",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

    print(f"Point cloud: {len(points_mb)} points")

---
## 6. Depth Precision Analysis

### 6.1 Precision vs Distance for Different Baselines

The depth precision formula $\Delta Z = \frac{Z^2}{fB} \Delta d$ tells us that:

1. **Quadratic growth**: error grows as $Z^2$, not $Z$
2. **Baseline matters**: doubling $B$ halves the depth error
3. **Sub-pixel accuracy matters**: $\Delta d = 0.25$ px vs $0.5$ px halves the error
4. **Focal length matters**: longer $f$ → better precision (narrower FOV)

### 6.2 Maximum Usable Range

Given a target maximum relative error $\epsilon = \Delta Z / Z$:

$$
\epsilon = \frac{Z}{fB} \Delta d \implies Z_{\max} = \frac{\epsilon \cdot f \cdot B}{\Delta d}
$$

For 10% relative accuracy ($\epsilon = 0.1$):

| $B$ (m) | $f$ (px) | $\Delta d$ (px) | $Z_{\max}$ (m) |
|:---:|:---:|:---:|:---:|
| 0.06 | 500 | 0.5 | 6.0 |
| 0.12 | 500 | 0.5 | 12.0 |
| 0.24 | 500 | 0.5 | 24.0 |
| 0.54 | 500 | 0.5 | 54.0 |
| 0.12 | 500 | 0.25 | 24.0 |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

Z_range = np.linspace(0.5, 80, 500)
delta_d_vals = [0.25, 0.5, 1.0]

ax = axes[0]
for B_val, ls in [(0.06, '--'), (0.12, '-'), (0.24, '-.'), (0.54, ':')]:
    dZ = depth_precision(Z_range, f, B_val, 0.5)
    ax.semilogy(Z_range, dZ, ls, linewidth=2, label=f'B = {B_val} m')
ax.axhline(1.0, color='gray', alpha=0.5, linestyle='--')
ax.set_xlabel('Distance Z (m)')
ax.set_ylabel(r'$\Delta Z$ (m)')
ax.set_title(r'Depth Error vs Distance ($\Delta d = 0.5$ px)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(1e-3, 100)

ax = axes[1]
for dd, ls in [(0.1, ':'), (0.25, '--'), (0.5, '-'), (1.0, '-.')]:
    dZ = depth_precision(Z_range, f, B, dd)
    ax.semilogy(Z_range, dZ, ls, linewidth=2, label=f'Δd = {dd} px')
ax.axhline(1.0, color='gray', alpha=0.5, linestyle='--')
ax.set_xlabel('Distance Z (m)')
ax.set_ylabel(r'$\Delta Z$ (m)')
ax.set_title(r'Depth Error vs Sub-pixel Accuracy ($B = 0.12$ m)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(1e-3, 100)

ax = axes[2]
for B_val, ls in [(0.06, '--'), (0.12, '-'), (0.24, '-.'), (0.54, ':')]:
    rel_err = Z_range / (f * B_val) * 0.5  # ΔZ/Z = Z·Δd/(fB)
    ax.semilogy(Z_range, rel_err * 100, ls, linewidth=2, label=f'B = {B_val} m')
ax.axhline(10, color='red', alpha=0.5, linestyle='--', label='10% threshold')
ax.set_xlabel('Distance Z (m)')
ax.set_ylabel('Relative Error ΔZ/Z (%)')
ax.set_title(r'Relative Depth Error ($\Delta d = 0.5$ px)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.1, 1000)

plt.tight_layout()
plt.show()

print("\n=== Maximum Usable Range (10% relative error) ===")
epsilon = 0.1
for B_val in [0.06, 0.12, 0.24, 0.54]:
    for dd in [0.25, 0.5]:
        z_max = epsilon * f * B_val / dd
        print(f"B = {B_val:.2f} m, Δd = {dd:.2f} px  →  Z_max = {z_max:.1f} m")

## 7. Learned Stereo Matching

### 7.1 Evolution of Deep Stereo

Deep stereo matching has evolved through three generations:

**Generation 1 — Cost Volume + 3D CNN (2017–2020)**. GC-Net and PSMNet build a 4D cost volume
$C(d, h, w)$ from concatenated left/right features and filter it with 3D convolutions. Accurate but
memory-hungry: $O(D \times H \times W)$ for max disparity $D$.

**Generation 2 — Iterative Refinement (2020–2024)**. RAFT-Stereo (Lipson et al., ECCV 2022) adapts
RAFT's correlation lookup and GRU-based iterative update to stereo. The key insight is replacing
the 4D cost volume with a 3D correlation pyramid and iteratively refining disparity:

$$d^{(k+1)} = d^{(k)} + \Delta d^{(k)}, \qquad \Delta d^{(k)} = \text{ConvGRU}\!\big(\text{context}, \text{corr\_lookup}(d^{(k)}), d^{(k)}\big)$$

This reduces memory to $O(H \times W)$ and enables arbitrary disparity ranges. CREStereo further
adds cascaded recurrence for sub-pixel accuracy. This family excels at cross-domain generalization
but requires multiple GRU iterations (typically 12–32).

**Generation 3 — Foundation Models (2025–2026)**. The newest paradigm trains on massive
synthetic datasets and incorporates monocular depth priors:

- **FoundationStereo** (Wen et al., CVPR 2025): trained on 1M synthetic pairs with automatic
  self-curation; uses DepthAnythingV2 as a frozen side-tuning backbone to inject monocular priors
  that bridge the sim-to-real gap; long-range context reasoning via transformer attention for cost
  filtering. Achieves new SOTA on zero-shot benchmarks across domains.
- **DEFOM-Stereo** (Jiang et al., CVPR 2025): injects monocular depth from a foundation model
  into RAFT-Stereo's recurrent updates; introduces a scale update module that aligns relative
  monocular depth to metric stereo disparity at each iteration. Ranks 1st on KITTI 2012/2015
  and Middlebury.
- **Fast-FoundationStereo** (Wen et al., CVPR 2026): distills FoundationStereo into a real-time
  model via (1) knowledge distillation of the hybrid backbone, (2) blockwise NAS for cost filtering,
  (3) structured pruning of the GRU refinement. 10× faster while closely matching teacher accuracy.
  Uses 1.4M pseudo-labeled in-the-wild stereo pairs for distillation.

### 7.2 Architectural Comparison

| Component | RAFT-Stereo | FoundationStereo | Fast-FoundationStereo |
|:---|:---|:---|:---|
| Feature backbone | CNN (ResNet) | CNN + frozen VFM (side-tuning) | Distilled single backbone |
| Cost volume | 1D correlation pyramid | 4D + long-range transformer | NAS-optimized blocks |
| Update mechanism | ConvGRU (multi-level) | ConvGRU + attention | Pruned ConvGRU |
| Training data | SceneFlow (35K) | 1M synthetic | 1M synthetic + 1.4M pseudo |
| Zero-shot accuracy | Good | **SOTA** | ~SOTA at 10× speed |
| Real-time? | No (12+ iterations) | No | **Yes** |

### 7.3 When to Use What

Classical SGM remains dominant on edge hardware (drones, FPGA systems) due to deterministic
latency, no training-data dependency, and roughly 10× lower power consumption.

**Recommendation for autonomous drones**:
- **Real-time obstacle avoidance**: classical SGBM or Fast-FoundationStereo
- **High-quality offline 3D reconstruction**: FoundationStereo or DEFOM-Stereo
- **No GPU available**: classical SGBM (runs on CPU/FPGA)
- **Monocular only** (no stereo baseline): monocular depth networks (Notebook 09)

---
## 8. Exercises

### Exercise 8.1: Implement the Stereo Equation

Complete the functions below to implement the stereo depth pipeline from scratch.

In [ ]:
def exercise_stereo_depth(disparity, focal_length, baseline):
    """
    Exercise 8.1a: Convert disparity to depth.
    
    Implement: Z = f·B / d
    Handle d <= 0 by setting those depths to 0.
    
    Parameters
    ----------
    disparity : np.ndarray (H, W) – disparity in pixels
    focal_length : float – focal length in pixels
    baseline : float – baseline in metres
    
    Returns
    -------
    depth : np.ndarray (H, W) – depth in metres
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement stereo_depth")


def exercise_depth_precision(Z, focal_length, baseline, delta_d):
    """
    Exercise 8.1b: Compute depth precision.
    
    Implement: ΔZ = Z² / (f·B) · Δd
    
    Parameters
    ----------
    Z : float or np.ndarray – depth(s) in metres
    focal_length : float – focal length in pixels
    baseline : float – baseline in metres
    delta_d : float – disparity uncertainty in pixels
    
    Returns
    -------
    delta_Z : same shape as Z – depth error in metres
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement depth_precision")


# --- Tests ---
# Uncomment after implementing:
# d_test = np.array([[12.0, 6.0], [3.0, 0.0]])
# z_test = exercise_stereo_depth(d_test, 500.0, 0.12)
# assert np.allclose(z_test[0, 0], 5.0), f"Expected 5.0, got {z_test[0, 0]}"
# assert z_test[1, 1] == 0.0, "Zero disparity should give zero depth"
# print("✓ exercise_stereo_depth passed")
#
# dz = exercise_depth_precision(10.0, 500.0, 0.12, 0.5)
# assert abs(dz - 0.8333) < 0.01, f"Expected ~0.833, got {dz}"
# print("✓ exercise_depth_precision passed")

### Exercise 8.2: Compute Disparity and Build a Point Cloud

Use the synthetic stereo pair to:
1. Compute the SGBM disparity map
2. Convert to depth
3. Back-project into a 3-D point cloud
4. Visualize the result

In [ ]:
def exercise_full_pipeline(img_l, img_r, K, baseline, num_disparities=128,
                           block_size=5, max_depth=15.0):
    """
    Exercise 8.2: Full stereo-to-pointcloud pipeline.
    
    Steps:
    1. Compute SGBM disparity
    2. Convert disparity to depth: Z = f·B / d
    3. Back-project each valid pixel to 3D:
       X = (u - cx) · Z / fx
       Y = (v - cy) · Z / fy
    
    Returns
    -------
    disparity : np.ndarray (H, W)
    depth : np.ndarray (H, W)
    points_3d : np.ndarray (N, 3)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement full pipeline")


# --- Run after implementing ---
# disparity, depth, points = exercise_full_pipeline(img_left, img_right, K, B)
# print(f"Disparity range: [{disparity[disparity>0].min():.1f}, {disparity.max():.1f}] px")
# print(f"Depth range:     [{depth[depth>0].min():.2f}, {depth[depth>0].max():.2f}] m")
# print(f"Point cloud:     {len(points)} points")

### Exercise 8.3: Depth Precision Study

Plot $\Delta Z$ vs distance for the following configurations and determine the
maximum range where relative error stays below 5%:

1. Indoor robot: $f = 400$ px, $B = 0.06$ m, $\Delta d = 0.5$ px
2. Autonomous car: $f = 700$ px, $B = 0.54$ m, $\Delta d = 0.25$ px
3. Drone mapping: $f = 500$ px, $B = 0.20$ m, $\Delta d = 0.5$ px

In [ ]:
# Exercise 8.3: Implement your precision analysis here

configs = [
    {"name": "Indoor Robot",   "f": 400, "B": 0.06, "dd": 0.5},
    {"name": "Autonomous Car", "f": 700, "B": 0.54, "dd": 0.25},
    {"name": "Drone Mapping",  "f": 500, "B": 0.20, "dd": 0.5},
]

# YOUR CODE HERE:
# 1. For each configuration, compute ΔZ over Z = [0.5, 100] m
# 2. Compute relative error ΔZ/Z
# 3. Find Z_max where relative error = 5%
# 4. Plot all three on the same axes
# 
# Hint: Z_max = epsilon * f * B / delta_d

---

### Exercise 8.5: Simple Block Matching (SAD) from Scratch

Implement the simplest stereo matching algorithm — Sum of Absolute Differences
block matching. Compare quality against SGBM and observe failures in
textureless regions.

In [ ]:
def block_matching_sad(left: np.ndarray, right: np.ndarray,
                       block_size: int = 5, max_disp: int = 64) -> np.ndarray:
    """Compute disparity map using Sum of Absolute Differences (SAD).
    
    Parameters
    ----------
    left, right : (H, W) uint8 arrays
        Rectified stereo pair (grayscale).
    block_size : int
        Window size (must be odd).
    max_disp : int
        Maximum disparity to search.
    
    Returns
    -------
    disparity : (H, W) float32 array
        Disparity map (pixels).
    """
    assert block_size % 2 == 1, "block_size must be odd"
    h, w = left.shape
    half = block_size // 2
    disparity = np.zeros((h, w), dtype=np.float32)
    
    left_f = left.astype(np.float32)
    right_f = right.astype(np.float32)
    
    for y in range(half, h - half):
        for x in range(half + max_disp, w - half):
            patch_l = left_f[y-half:y+half+1, x-half:x+half+1]
            best_d, best_cost = 0, np.inf
            for d in range(max_disp):
                patch_r = right_f[y-half:y+half+1, x-d-half:x-d+half+1]
                cost = np.sum(np.abs(patch_l - patch_r))
                if cost < best_cost:
                    best_cost = cost
                    best_d = d
            disparity[y, x] = best_d
    return disparity

np.random.seed(42)
H, W = 60, 120
left_img = np.random.randint(0, 256, (H, W), dtype=np.uint8)
true_disp = 10
right_img = np.zeros_like(left_img)
right_img[:, :W-true_disp] = left_img[:, true_disp:]

disp = block_matching_sad(left_img, right_img, block_size=5, max_disp=20)
center_disp = disp[H//2, W//2]
print(f"Estimated disparity at center: {center_disp:.0f} (expected: {true_disp})")
assert abs(center_disp - true_disp) <= 2, f"Disparity error too large: {abs(center_disp - true_disp)}"

---

### Exercise 8.6: Left-Right Consistency Check

Implement and visualize the left-right consistency check for disparity
validation. Identify occluded and unreliable regions.

In [ ]:
def lr_consistency_check(disp_left: np.ndarray, disp_right: np.ndarray,
                         threshold: float = 1.0) -> np.ndarray:
    """Left-right consistency check for disparity maps.
    
    A pixel passes if |d_L(x, y) - d_R(x - d_L(x,y), y)| <= threshold.
    
    Parameters
    ----------
    disp_left : (H, W) float32
        Left disparity map.
    disp_right : (H, W) float32
        Right disparity map.
    threshold : float
        Maximum allowed inconsistency (pixels).
    
    Returns
    -------
    valid_mask : (H, W) bool array
        True where the disparity is consistent.
    """
    h, w = disp_left.shape
    valid_mask = np.zeros((h, w), dtype=bool)
    
    for y in range(h):
        for x in range(w):
            d = int(round(disp_left[y, x]))
            x_r = x - d
            if 0 <= x_r < w:
                if abs(disp_left[y, x] - disp_right[y, x_r]) <= threshold:
                    valid_mask[y, x] = True
    return valid_mask

H, W = 50, 100
disp_L = np.full((H, W), 8.0, dtype=np.float32)
disp_R = np.full((H, W), 8.0, dtype=np.float32)

disp_R[:, 40:60] = 20.0  # wrong disparities in right map

mask = lr_consistency_check(disp_L, disp_R, threshold=1.0)
consistent_pct = mask.sum() / mask.size * 100
print(f"Consistent pixels: {consistent_pct:.1f}%")
assert consistent_pct < 100, "Some pixels should be inconsistent"
assert consistent_pct > 50, "Most pixels should be consistent"
print(f"✓ LR check correctly identified {100-consistent_pct:.1f}% inconsistent pixels")

---
## Summary

### Key Equations

| Equation | Formula |
|:---|:---|
| Stereo depth | $Z = \dfrac{f \cdot B}{d}$ |
| Depth precision | $\Delta Z = \dfrac{Z^2}{f \cdot B} \cdot \Delta d$ |
| 3D back-projection | $X = \dfrac{(u - c_x) \cdot Z}{f}$ |
| Max usable range | $Z_{\max} = \dfrac{\epsilon \cdot f \cdot B}{\Delta d}$ |
| SGBM energy | $E(D) = \sum_p C(p, D_p) + \sum_{q \in N_p} P_1 \mathbb{1}_{\Delta D=1} + P_2 \mathbb{1}_{\Delta D>1}$ |

### Key Takeaways

1. Stereo depth is the ratio of focal length × baseline to disparity
2. Depth error grows **quadratically** with distance — stereo is most useful at close range
3. SGBM approximates global MRF optimisation via 8-direction 1-D DP
4. Census transform provides illumination-invariant matching
5. The baseline–accuracy tradeoff determines the useful operating range

### What's Next

→ **Notebook 09**: Neural depth estimation — when a single camera must estimate depth